In [ ]:
import numpy as np
from fake_spectra.griddedspectra import GriddedSpectra
from fake_spectra.spectra import Spectra
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [ ]:
# Compute raw flux power spectra (before normalization) and plot ratio
# Low-res (keir, Ngrid=512):  snap 0->z=5.0, 1->z=4.6, 2->z=4.2
# High-res (keir_highres, Ngrid=1024): snap 3->z=5.0, 4->z=4.6, 5->z=4.2

redshifts = [5.0, 4.6, 4.2]
low_snaps = [0, 1, 2]
high_snaps = [3, 4, 5]
nspec = 200

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, (z, ls, hs) in enumerate(zip(redshifts, low_snaps, high_snaps)):
    # Low-res raw flux power (no mean_flux_desired rescaling)
    gs_low = Spectra(ls, "keir/output", MPI=None, res=1.0,
                     savefile="gridded_spectra_%03d.hdf5" % nspec,
                     cofm=None, axis=None)
    fps_low = gs_low.get_flux_power_1D("H", 1)
    k_low, pf_low = fps_low[0], fps_low[1]

    # High-res raw flux power (no mean_flux_desired rescaling)
    gs_high = Spectra(hs, "keir_highres/output", MPI=None, res=1.0,
                      savefile="gridded_spectra_%03d.hdf5" % nspec,
                      cofm=None, axis=None)
    fps_high = gs_high.get_flux_power_1D("H", 1)
    k_high, pf_high = fps_high[0], fps_high[1]

    # Interpolate high-res onto low-res k bins for the ratio
    k_min = max(k_low.min(), k_high.min())
    k_max = min(k_low.max(), k_high.max())
    mask_low = (k_low >= k_min) & (k_low <= k_max)
    k_common = k_low[mask_low]

    interp_high = interp1d(k_high, pf_high, kind='linear',
                           bounds_error=False, fill_value="extrapolate")
    pf_high_rebinned = interp_high(k_common)
    pf_low_common = pf_low[mask_low]

    ratio = pf_high_rebinned / pf_low_common

    ax = axes[idx]
    ax.plot(k_common, ratio, label=f"z = {z:.1f}")
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("k (s/km)")
    ax.set_title(f"z = {z:.1f}")
    ax.legend()

axes[0].set_ylabel(r"$P_F^{\rm high}(k) / P_F^{\rm low}(k)$")
plt.suptitle("Flux Power Spectrum Ratio: High-res (1024) / Low-res (512), before normalization")
plt.tight_layout()
plt.show()